# 4053. Minimum Operations to Make Every Element Palindromic — Nhập môn "Sinh trước tất cả ứng viên + Binary Search"

Bài toán LeetCode (Medium):

```python
class Solution:
    def minOperations(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Vì sao phép "cộng/trừ 2" **giữ nguyên tính chẵn lẻ**, và hệ quả của nó.
- Vì sao có thể xử lý **từng phần tử độc lập**.
- Kiểm tra số palindrome bằng chuỗi: `str(x)`, slicing `s[::-1]`.
- Lời giải Brute Force "đi dần ra hai phía" và vì sao nó có thể cực chậm.
- Nhận xét then chốt: **số palindrome rất ít** (~120 nghìn số dưới 2·10^9) → **sinh trước tất cả**.
- Cách **sinh palindrome bằng cách lật nửa đầu** (mirroring).
- **Binary search** bằng `bisect` để tìm palindrome gần nhất.
- Cách thứ ba: **sắp xếp + hai con trỏ** (xử lý offline).
- Kỹ thuật **tính trước một lần** (precompute) dùng chung cho mọi test.
- Lỗi hay mắc, biến thể, bài tập có đáp án.

# 1. Đề bài nói gì?

Cho mảng số nguyên `nums`. Mỗi thao tác: chọn một chỉ số `i` và **tăng hoặc giảm `nums[i]` đi 2**.

Trả về số thao tác **ít nhất** để **mọi** phần tử đều trở thành **số palindrome dương**. Các phần tử có thể thành các palindrome khác nhau.

> **Palindrome** (số đối xứng): đọc xuôi hay ngược đều như nhau, ví dụ `7`, `22`, `121`, `9009`.

### Ví dụ

| nums | Kết quả | Giải thích |
|---|---:|---|
| `[10, 12, 14, 16]` | 9 | `10→8` (1), `12→8` (2), `14→8` (3), `16→22` (3) |
| `[9, 10, 11, 10]` | 2 | `9`, `11` đã là palindrome; `10→8` hai lần (1 + 1) |
| `[125]` | 2 | `125→123→121` |

### Ràng buộc

- `1 <= n <= 10^5`
- `1 <= nums[i] <= 10^9`

# 2. Quan sát 1 — Cộng/trừ 2 không đổi tính chẵn lẻ

`x + 2` và `x - 2` có cùng tính chẵn lẻ với `x`. Vậy:

- Số **chẵn** chỉ có thể biến thành số **chẵn**.
- Số **lẻ** chỉ có thể biến thành số **lẻ**.

Ví dụ `10` (chẵn) **không thể** thành `9` hay `11` (palindrome lẻ), dù chúng chỉ cách 1 đơn vị. Nó phải về `8` hoặc lên `22`.

Từ `x` đến `p` (cùng chẵn lẻ) mất đúng `|x - p| / 2` thao tác.

# 3. Quan sát 2 — Các phần tử độc lập với nhau

Mỗi thao tác chỉ đụng **một** phần tử, và yêu cầu cho mỗi phần tử ("thành palindrome") không phụ thuộc phần tử khác.

> Đáp án = **tổng** chi phí nhỏ nhất của từng phần tử.

```text
cost(x) = min |x - p| / 2   với p là palindrome dương, p cùng chẵn lẻ với x
```

Bài toán thu gọn thành: **với mỗi `x`, tìm palindrome cùng chẵn lẻ gần `x` nhất.**

# 4. Nền tảng Python — Kiểm tra palindrome bằng chuỗi

- `str(121)` → chuỗi `"121"`.
- Slicing `s[start:end:step]`; với `step = -1` và bỏ trống start/end, `s[::-1]` là **chuỗi đảo ngược**.
- So sánh `s == s[::-1]` để kiểm tra đối xứng.

In [1]:
s = str(1231)
print(s, "->", s[::-1])

def is_palindrome(x):
    s = str(x)
    return s == s[::-1]

for x in [7, 10, 22, 121, 123, 9009]:
    print(x, is_palindrome(x))

1231 -> 1321
7 True
10 False
22 True
121 True
123 False
9009 True


# 5. Cách 1 — Brute Force: đi dần ra hai phía

Với mỗi `x`, thử `d = 0, 2, 4, ...`: nếu `x - d` (phải `> 0`) hoặc `x + d` là palindrome thì dừng, chi phí `d / 2`.

Vì `d` tăng dần, palindrome tìm thấy đầu tiên chắc chắn là **gần nhất**.

In [2]:
def nearest_cost_bruteforce(x):
    d = 0
    while True:
        if x - d > 0 and is_palindrome(x - d):
            return d // 2
        if is_palindrome(x + d):
            return d // 2
        d += 2


def min_operations_bruteforce(nums):
    return sum(nearest_cost_bruteforce(x) for x in nums)


print(min_operations_bruteforce([10, 12, 14, 16]))   # 9
print(min_operations_bruteforce([9, 10, 11, 10]))    # 2
print(min_operations_bruteforce([125]))              # 2

9
2
2


## Vì sao Brute Force có thể cực chậm?

Palindrome **chẵn** phải bắt đầu và kết thúc bằng chữ số chẵn. Với số 9 chữ số, palindrome chẵn lớn nhất là `899999998`; palindrome chẵn tiếp theo phải đợi tới 10 chữ số: `2000000002`.

Vậy với `x = 1 000 000 000` (chẵn), palindrome chẵn gần nhất là `899999998`, cách hơn **100 triệu** → vòng `while` chạy khoảng **50 triệu** lần, cho **một** phần tử. Với `10^5` phần tử như vậy thì không thể.

Ta chỉ đo **khoảng cách** (không chạy brute force thật) để thấy:

In [3]:
x = 10**9
print("899999998 là palindrome chẵn?", is_palindrome(899999998), 899999998 % 2 == 0)
print("Khoảng cách tới 899999998:", x - 899999998, "-> số vòng lặp brute force ≈", (x - 899999998) // 2)

899999998 là palindrome chẵn? True True
Khoảng cách tới 899999998: 100000002 -> số vòng lặp brute force ≈ 50000001


# 6. Quan sát 3 — Palindrome rất hiếm!

Một palindrome được **xác định hoàn toàn bởi nửa đầu** của nó:

- `12321` ← nửa đầu `123` (lật `12` ghép vào)
- `123321` ← nửa đầu `123` (lật `123` ghép vào)

Số palindrome có `L` chữ số = số cách chọn nửa đầu = `9 × 10^(⌈L/2⌉ - 1)` (chữ số đầu khác 0).

| Số chữ số L | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| Số palindrome | 9 | 9 | 90 | 90 | 900 | 900 | 9 000 | 9 000 | 90 000 | 90 000 |

Tổng cộng dưới `10^10` chỉ khoảng **200 nghìn** số — quá ít so với `10^10` số nguyên!

> **Ý tưởng:** sinh **trước** mọi palindrome cần thiết, sắp xếp, rồi với mỗi `x` dùng **binary search** tìm hàng xóm gần nhất.

Ta cần sinh tới đâu? Phải đủ để mọi `x <= 10^9` có ứng viên **ở cả hai phía**. Palindrome đầu tiên lớn hơn `10^9` là `1000000001` (lẻ), palindrome chẵn đầu tiên lớn hơn `10^9` là `2000000002`. Sinh tới `2 000 000 002` là đủ an toàn.

# 7. Sinh palindrome bằng cách lật nửa đầu

Với nửa đầu `h` (dạng chuỗi):

- **Độ dài chẵn:** `h + h[::-1]` → `"123" + "321" = "123321"`
- **Độ dài lẻ:** `h + h[-2::-1]` → `"123" + "21" = "12321"` (bỏ chữ số cuối của `h` để không lặp)

`h[-2::-1]` nghĩa là: bắt đầu từ ký tự **áp chót** (`-2`), đi **ngược** (`-1`) về đầu.

In [4]:
h = "123"
print("h[::-1]   =", h[::-1])
print("h[-2::-1] =", h[-2::-1])
print("chẵn:", h + h[::-1], "| lẻ:", h + h[-2::-1])

h[::-1]   = 321
h[-2::-1] = 21
chẵn: 123321 | lẻ: 12321


In [5]:
def generate_palindromes(limit):
    """Mọi palindrome dương <= limit, theo thứ tự tăng dần."""
    result = []
    length = 1
    while True:
        half_len = (length + 1) // 2
        smallest = int("1" + "0" * (length - 1))       # số nhỏ nhất có `length` chữ số
        if smallest > limit:
            break
        for half in range(10 ** (half_len - 1), 10 ** half_len):
            h = str(half)
            if length % 2 == 0:
                p = int(h + h[::-1])
            else:
                p = int(h + h[-2::-1])
            if p > limit:
                break                                  # các half sau còn lớn hơn
            result.append(p)
        length += 1
    return result


small = generate_palindromes(300)
print(small)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 22, 33, 44, 55, 66, 77, 88, 99, 101, 111, 121, 131, 141, 151, 161, 171, 181, 191, 202, 212, 222, 232, 242, 252, 262, 272, 282, 292]


Vì sao danh sách tự **tăng dần**? Ta sinh theo **độ dài tăng dần**, và trong cùng độ dài, nửa đầu tăng thì palindrome tăng. Nhờ vậy `break` khi `p > limit` là an toàn.

Bây giờ sinh thật với `limit = 2 000 000 002`, rồi tách theo chẵn/lẻ.

In [6]:
import time

LIMIT = 2_000_000_002
t0 = time.time()
ALL_PAL = generate_palindromes(LIMIT)
EVEN_PAL = [p for p in ALL_PAL if p % 2 == 0]
ODD_PAL = [p for p in ALL_PAL if p % 2 == 1]
print(f"Sinh {len(ALL_PAL)} palindrome trong {time.time() - t0:.2f} giây")
print("chẵn:", len(EVEN_PAL), EVEN_PAL[:6], "...", EVEN_PAL[-3:])
print("lẻ  :", len(ODD_PAL), ODD_PAL[:6], "...", ODD_PAL[-3:])

Sinh 119999 palindrome trong 0.03 giây
chẵn: 48889 [2, 4, 6, 8, 22, 44] ... [899989998, 899999998, 2000000002]
lẻ  : 71110 [1, 3, 5, 7, 9, 11] ... [1999779991, 1999889991, 1999999991]


Viết `2_000_000_002`: Python cho phép dùng dấu gạch dưới `_` để **phân nhóm chữ số** cho dễ đọc; giá trị không đổi.

# 8. Binary Search tìm hàng xóm gần nhất

Trong danh sách tăng dần `arr`, `bisect_left(arr, x)` trả về vị trí `i` đầu tiên có `arr[i] >= x`. Khi đó:

- `arr[i]` (nếu `i < len(arr)`) là palindrome **nhỏ nhất `>= x`** — hàng xóm bên phải.
- `arr[i - 1]` (nếu `i > 0`) là palindrome **lớn nhất `< x`** — hàng xóm bên trái.

Palindrome gần nhất chắc chắn là một trong hai số này.

```text
arr:   ...   8      22     44  ...
                 ^ x = 10
       arr[i-1] = 8,  arr[i] = 22   -> gần nhất là 8 (khoảng cách 2)
```

In [7]:
from bisect import bisect_left

x = 10
i = bisect_left(EVEN_PAL, x)
print("i =", i, "| trái =", EVEN_PAL[i - 1], "| phải =", EVEN_PAL[i])

x = 125
i = bisect_left(ODD_PAL, x)
print("i =", i, "| trái =", ODD_PAL[i - 1], "| phải =", ODD_PAL[i])

i = 4 | trái = 8 | phải = 22
i = 13 | trái = 121 | phải = 131


# 9. Cách 2 — Sinh trước + Binary Search

```text
với mỗi x:
    arr = EVEN_PAL nếu x chẵn, ngược lại ODD_PAL
    i = bisect_left(arr, x)
    ứng viên: arr[i] (nếu có), arr[i-1] (nếu có)
    cost = min khoảng cách / 2
```

Không cần lo "palindrome dương": danh sách chỉ chứa số `>= 1`.

In [8]:
def nearest_cost_bisect(x):
    arr = EVEN_PAL if x % 2 == 0 else ODD_PAL
    i = bisect_left(arr, x)
    best = float("inf")
    if i < len(arr):
        best = min(best, arr[i] - x)
    if i > 0:
        best = min(best, x - arr[i - 1])
    return best // 2


def min_operations_bisect(nums):
    return sum(nearest_cost_bisect(x) for x in nums)


print(min_operations_bisect([10, 12, 14, 16]))   # 9
print(min_operations_bisect([9, 10, 11, 10]))    # 2
print(min_operations_bisect([125]))              # 2
print(nearest_cost_bisect(10**9))                # cực nhanh dù brute force sẽ rất chậm

9
2
2
50000001


# 10. Mô phỏng từng bước

Ví dụ `nums = [10, 12, 14, 16]` — mọi số đều chẵn, dùng `EVEN_PAL = [2, 4, 6, 8, 22, 44, ...]`:

| x | `bisect_left` | trái | phải | khoảng cách trái | khoảng cách phải | chọn | thao tác |
|---|---:|---:|---:|---:|---:|---|---:|
| 10 | 4 | 8 | 22 | 2 | 12 | 8 | 1 |
| 12 | 4 | 8 | 22 | 4 | 10 | 8 | 2 |
| 14 | 4 | 8 | 22 | 6 | 8 | 8 | 3 |
| 16 | 4 | 8 | 22 | 8 | 6 | 22 | 3 |

Tổng `= 9` ✔.

In [9]:
def trace_min_operations(nums):
    total = 0
    for x in nums:
        arr, kind = (EVEN_PAL, "chẵn") if x % 2 == 0 else (ODD_PAL, "lẻ")
        i = bisect_left(arr, x)
        left = arr[i - 1] if i > 0 else None
        right = arr[i] if i < len(arr) else None
        options = []
        if left is not None:
            options.append((x - left, left))
        if right is not None:
            options.append((right - x, right))
        dist, target = min(options)
        total += dist // 2
        print(f"x={x:>4} ({kind}): trái={left}, phải={right} -> chọn {target}, "
              f"khoảng cách {dist}, thao tác {dist // 2}, tổng = {total}")
    return total

trace_min_operations([10, 12, 14, 16])
print()
trace_min_operations([9, 10, 11, 10, 125])

x=  10 (chẵn): trái=8, phải=22 -> chọn 8, khoảng cách 2, thao tác 1, tổng = 1
x=  12 (chẵn): trái=8, phải=22 -> chọn 8, khoảng cách 4, thao tác 2, tổng = 3
x=  14 (chẵn): trái=8, phải=22 -> chọn 8, khoảng cách 6, thao tác 3, tổng = 6
x=  16 (chẵn): trái=8, phải=22 -> chọn 22, khoảng cách 6, thao tác 3, tổng = 9

x=   9 (lẻ): trái=7, phải=9 -> chọn 9, khoảng cách 0, thao tác 0, tổng = 0
x=  10 (chẵn): trái=8, phải=22 -> chọn 8, khoảng cách 2, thao tác 1, tổng = 1
x=  11 (lẻ): trái=9, phải=11 -> chọn 11, khoảng cách 0, thao tác 0, tổng = 1
x=  10 (chẵn): trái=8, phải=22 -> chọn 8, khoảng cách 2, thao tác 1, tổng = 2
x= 125 (lẻ): trái=121, phải=131 -> chọn 121, khoảng cách 4, thao tác 2, tổng = 4


4

Ở cell trên, `min(options)` với `options` là list các **tuple** `(khoảng_cách, palindrome)`: Python so sánh tuple từ trái sang, nên lấy được tuple có khoảng cách nhỏ nhất. Đây là mẹo "lấy min kèm thông tin".

Trường hợp `x` đã là palindrome cùng chẵn lẻ (ví dụ `9`, `11`): `bisect_left` trả về đúng vị trí của `x`, nên `arr[i] = x` và khoảng cách `0`.

# 11. Cách 3 — Sắp xếp + hai con trỏ (xử lý offline)

Một cách khác không cần `bisect`: nếu ta **sắp xếp** các `x` (cùng chẵn lẻ) tăng dần, thì vị trí "hàng xóm" trong danh sách palindrome cũng chỉ **tiến về phía trước**. Dùng một con trỏ `ptr` chạy dần trên danh sách palindrome:

```text
với mỗi x (đã sắp xếp tăng dần):
    while ptr < len(arr) and arr[ptr] < x:  ptr += 1
    -> arr[ptr] là palindrome nhỏ nhất >= x, arr[ptr-1] là lớn nhất < x
```

Vì `ptr` không bao giờ lùi lại, tổng số bước của nó `<= len(arr)`. Đây là kỹ thuật **hai con trỏ** (two pointers) — rất hay dùng khi cả hai dãy đều đã sắp xếp.

In [10]:
def min_operations_two_pointers(nums):
    total = 0
    for parity, arr in ((0, EVEN_PAL), (1, ODD_PAL)):
        xs = sorted(x for x in nums if x % 2 == parity)
        ptr = 0
        for x in xs:
            while ptr < len(arr) and arr[ptr] < x:
                ptr += 1
            best = float("inf")
            if ptr < len(arr):
                best = arr[ptr] - x
            if ptr > 0:
                best = min(best, x - arr[ptr - 1])
            total += best // 2
    return total


print(min_operations_two_pointers([10, 12, 14, 16]))   # 9
print(min_operations_two_pointers([9, 10, 11, 10]))    # 2
print(min_operations_two_pointers([125]))              # 2

9
2
2


So sánh cách 2 và cách 3:

- Cách 2: mỗi `x` tốn `O(log P)` (với `P ≈ 1.2·10^5` palindrome), không cần sắp xếp `nums`.
- Cách 3: tốn `O(n log n)` để sắp xếp `nums` cộng `O(P)` cho con trỏ. Không cần `bisect`.

Cả hai đều rất nhanh. Cách 2 ngắn gọn hơn nên được chọn làm bản nộp.

# 12. Kỹ thuật "tính trước một lần" (precompute)

LeetCode gọi `minOperations` **nhiều lần** (mỗi test một lần), nhưng danh sách palindrome **không phụ thuộc input**. Nếu sinh lại trong mỗi lần gọi, ta lãng phí thời gian.

Giải pháp: sinh ở **mức module** (ngoài class) hoặc làm **thuộc tính của class** — chỉ chạy **một lần** khi file được nạp.

In [11]:
from bisect import bisect_left

def _build_palindromes(limit=2_000_000_002):
    even, odd = [], []
    length = 1
    while 10 ** (length - 1) <= limit:
        half_len = (length + 1) // 2
        for half in range(10 ** (half_len - 1), 10 ** half_len):
            h = str(half)
            p = int(h + h[::-1]) if length % 2 == 0 else int(h + h[-2::-1])
            if p > limit:
                break
            (even if p % 2 == 0 else odd).append(p)
        length += 1
    return even, odd

_EVEN, _ODD = _build_palindromes()      # chạy MỘT lần khi nạp file


class Solution:
    def minOperations(self, nums: list[int]) -> int:
        total = 0
        for x in nums:
            arr = _EVEN if x % 2 == 0 else _ODD
            i = bisect_left(arr, x)
            best = arr[i] - x if i < len(arr) else float("inf")
            if i > 0:
                best = min(best, x - arr[i - 1])
            total += best // 2
        return total

Dòng `(even if p % 2 == 0 else odd).append(p)`: biểu thức trong ngoặc **chọn ra một trong hai list**, rồi `.append(p)` thêm vào list đó. Gọn hơn viết `if/else` hai nhánh.

In [12]:
sol = Solution()

tests = [
    ([10, 12, 14, 16], 9),
    ([9, 10, 11, 10], 2),
    ([125], 2),
    ([1], 0),                         # 1 đã là palindrome
    ([2, 4, 6, 8], 0),
    ([10], 1),                        # 10 -> 8
    ([20], 1),                        # 20 -> 22
    ([10**9], (10**9 - 899999998) // 2),
    ([999999999], 0),
]
for nums, expected in tests:
    result = sol.minOperations(nums)
    print(f"{str(nums):22} -> {result:>10} | expected {expected:>10} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[10, 12, 14, 16]       ->          9 | expected          9 | PASS
[9, 10, 11, 10]        ->          2 | expected          2 | PASS
[125]                  ->          2 | expected          2 | PASS
[1]                    ->          0 | expected          0 | PASS
[2, 4, 6, 8]           ->          0 | expected          0 | PASS
[10]                   ->          1 | expected          1 | PASS
[20]                   ->          1 | expected          1 | PASS
[1000000000]           ->   50000001 | expected   50000001 | PASS
[999999999]            ->          0 | expected          0 | PASS


In [13]:
import random

random.seed(53)
xs = list(range(1, 3000)) + random.sample(range(1, 10**6), 300)
for x in xs:
    assert nearest_cost_bisect(x) == nearest_cost_bruteforce(x)
print("Cách 2 khớp brute force trên", len(xs), "giá trị ✔")

for _ in range(300):
    nums = [random.randint(1, 10**9) for _ in range(random.randint(1, 20))]
    assert sol.minOperations(nums) == min_operations_bisect(nums) == min_operations_two_pointers(nums)
print("Cách 2, cách 3 và Solution khớp nhau trên số lớn ✔")

big = [random.randint(1, 10**9) for _ in range(10**5)]
t0 = time.time()
sol.minOperations(big)
print(f"n = 1e5 chạy trong {time.time() - t0:.3f} giây (không tính thời gian sinh palindrome một lần)")

Cách 2 khớp brute force trên 3299 giá trị ✔


Cách 2, cách 3 và Solution khớp nhau trên số lớn ✔
n = 1e5 chạy trong 0.041 giây (không tính thời gian sinh palindrome một lần)


# 13. Vì sao thuật toán đúng?

1. **Độc lập:** mỗi thao tác chỉ đổi một phần tử, mục tiêu của các phần tử không ràng buộc nhau → tổng chi phí nhỏ nhất = tổng chi phí nhỏ nhất của từng phần tử.
2. **Chẵn lẻ:** `x ± 2` giữ chẵn lẻ, nên chỉ palindrome cùng chẵn lẻ là đích hợp lệ; và mọi đích cùng chẵn lẻ `p` đều đạt được với đúng `|x - p| / 2` thao tác (đi thẳng một chiều). Không lo đi qua số `<= 0`: đi thẳng từ `x` đến `p` với `p >= 1` thì mọi giá trị trung gian nằm giữa `x` và `p`, đều dương.
3. **Hàng xóm gần nhất:** trong danh sách tăng dần, mọi phần tử bên trái `arr[i-1]` đều xa hơn hoặc bằng `arr[i-1]`, mọi phần tử bên phải `arr[i]` đều xa hơn hoặc bằng `arr[i]`. Vậy chỉ cần so hai số này.
4. **Đủ ứng viên:** danh sách chứa mọi palindrome tới `2 000 000 002`, nên mọi `x <= 10^9` luôn có hàng xóm bên phải cùng chẵn lẻ (`1000000001` hoặc `2000000002`), và palindrome xa hơn nữa không thể gần hơn hàng xóm đó.

# 14. Độ phức tạp

Gọi `n = len(nums)`, `P ≈ 1.2·10^5` là số palindrome sinh trước.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Brute Force | có thể tới `~5·10^7` bước **mỗi phần tử** | `O(1)` |
| 2. Sinh trước + bisect | `O(P)` một lần + `O(n log P)` | `O(P)` |
| 3. Sắp xếp + hai con trỏ | `O(P)` một lần + `O(n log n + P)` | `O(P + n)` |

# 15. Những lỗi thường gặp

## Lỗi 1 — Quên ràng buộc chẵn lẻ

Tìm palindrome gần nhất **bất kể chẵn lẻ**: với `x = 10` sẽ chọn `9` hoặc `11` (khoảng cách 1) → `1 // 2 = 0` thao tác. Sai! Đáp án đúng là 1 (về `8`).

## Lỗi 2 — Chỉ sinh palindrome tới `10^9`

Có thể thiếu hàng xóm bên phải cho các `x` gần `10^9`. Trong bài này may mắn hàng xóm trái luôn gần hơn, nhưng **đừng dựa vào may mắn** — luôn đảm bảo có ứng viên hai phía, và kiểm tra `i < len(arr)` trước khi truy cập.

## Lỗi 3 — `IndexError` khi `i == 0` hoặc `i == len(arr)`

Luôn kiểm tra biên trước khi đọc `arr[i]` và `arr[i - 1]`.

## Lỗi 4 — Sinh lại danh sách trong mỗi lần gọi hàm

Đúng nhưng chậm: mỗi test tốn thêm ~0.1–0.2 giây; hàng trăm test sẽ bị TLE.

## Lỗi 5 — Sinh palindrome độ dài lẻ bị lặp chữ số giữa

`h + h[::-1]` luôn cho độ dài **chẵn**. Muốn độ dài lẻ phải bỏ chữ số cuối: `h + h[-2::-1]`.

In [14]:
def wrong_ignore_parity(x):
    i = bisect_left(ALL_PAL, x)
    best = ALL_PAL[i] - x
    if i > 0:
        best = min(best, x - ALL_PAL[i - 1])
    return best // 2

print("x = 10 | Sai (bỏ qua chẵn lẻ):", wrong_ignore_parity(10), "| Đúng:", nearest_cost_bisect(10))
print("Lỗi 5 :", "123" + "123"[::-1], "(độ dài 6, không phải palindrome 5 chữ số)")

x = 10 | Sai (bỏ qua chẵn lẻ): 0 | Đúng: 1
Lỗi 5 : 123321 (độ dài 6, không phải palindrome 5 chữ số)


# 16. Mẹo nhận diện pattern

- Đề cần "tìm số **đặc biệt** gần nhất" (palindrome, số nguyên tố, số chính phương, số Fibonacci, lũy thừa của 2...) → hỏi: **có bao nhiêu số đặc biệt trong miền?** Nếu ít (vài trăm nghìn) → **sinh trước + bisect**.
- Thao tác `±k` → chú ý **phần dư theo `k`** (ở đây `k = 2`: chẵn lẻ) là bất biến.
- Thao tác chỉ đụng **một phần tử** → tách bài thành các bài con độc lập rồi cộng.
- Dữ liệu tính trước không phụ thuộc input → đặt ở **mức module/class**.

# 17. Biến thể — Thao tác `±1` (không còn ràng buộc chẵn lẻ)

Nếu mỗi thao tác cộng/trừ **1**, mọi palindrome đều là đích hợp lệ và chi phí là `|x - p|`. Chỉ cần bisect trên `ALL_PAL`.

In [15]:
def min_operations_plus_minus_one(nums):
    total = 0
    for x in nums:
        i = bisect_left(ALL_PAL, x)
        best = ALL_PAL[i] - x
        if i > 0:
            best = min(best, x - ALL_PAL[i - 1])
        total += best
    return total

print(min_operations_plus_minus_one([10, 12, 14, 16]))   # 10->9 (1), 12->11 (1), 14->11 (3), 16->11 (5): tổng 10

10


# 18. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tìm palindrome cùng chẵn lẻ gần nhất và số thao tác cho: `10`, `21`, `100`, `125`, `1000`.

## Bài 2 — Dự đoán output

```python
"2468"[::-1], "2468"[-2::-1], "5"[-2::-1]
bisect_left([2, 4, 6, 8, 22], 8), bisect_left([2, 4, 6, 8, 22], 9)
```

In [16]:
print("2468"[::-1], "2468"[-2::-1], repr("5"[-2::-1]))
print(bisect_left([2, 4, 6, 8, 22], 8), bisect_left([2, 4, 6, 8, 22], 9))

8642 642 ''
3 4


## Bài 3 — Điền chỗ trống: palindrome có đúng `L` chữ số

In [17]:
def exercise_3_palindromes_of_length(L):
    result = []
    # TODO 1: half_len = (L + 1) // 2
    # TODO 2: duyệt half từ 10**(half_len-1) đến 10**half_len - 1
    # TODO 3: L chẵn: h + h[::-1] ; L lẻ: h + h[-2::-1]
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_palindromes_of_length(2) == [11, 22, 33, 44, 55, 66, 77, 88, 99]
# assert len(exercise_3_palindromes_of_length(5)) == 900

## Bài 4 — Tự viết `nearest_cost` bằng bisect

Viết lại hàm tính chi phí cho một phần tử, chú ý biên `i == 0` và `i == len(arr)`.

In [18]:
def exercise_4_nearest_cost(x, even_pal, odd_pal):
    # TODO
    raise NotImplementedError

## Bài 5 — Biến thể: thao tác `±3`

Mỗi thao tác cộng/trừ **3**. Bất biến giờ là gì? Viết hàm tính tổng thao tác ít nhất (đích vẫn là palindrome dương).

In [19]:
def exercise_5_plus_minus_three(nums):
    # TODO: nhóm palindrome theo p % 3
    raise NotImplementedError

## Bài 6 — Mini challenge: số nguyên tố gần nhất

Với mỗi `x` (`1 <= x <= 10^6`), tìm số thao tác `±1` ít nhất để `x` thành **số nguyên tố**. Gợi ý: sàng Eratosthenes sinh trước mọi số nguyên tố tới `10^6 + 100`, rồi bisect.

In [20]:
def exercise_6_nearest_prime_cost(nums):
    # TODO
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 3**
- Mức 1: palindrome `L` chữ số ↔ nửa đầu có `⌈L/2⌉` chữ số, chữ số đầu khác 0.
- Mức 2: `range(10 ** (half_len - 1), 10 ** half_len)`.
- Mức 3: `int(h + h[::-1])` hoặc `int(h + h[-2::-1])`.

**Bài 5**
- Mức 1: `x ± 3` giữ nguyên `x % 3`.
- Mức 2: chia `ALL_PAL` thành 3 list theo `p % 3`.
- Mức 3: chi phí `|x - p| / 3`, bisect trong list có cùng `x % 3`.

**Bài 6**
- Mức 1: sàng: `is_prime = [True] * (N+1)`, gạch bội của mỗi số nguyên tố từ `p*p`.
- Mức 2: `primes = [i for i in range(2, N+1) if is_prime[i]]`.
- Mức 3: giống hệt cách 2, nhưng chi phí là `|x - p|`.

# 19. Đáp án tham khảo

**Bài 1:**

| x | chẵn/lẻ | trái | phải | chọn | thao tác |
|---|---|---:|---:|---:|---:|
| 10 | chẵn | 8 | 22 | 8 | 1 |
| 21 | lẻ | 11 (cách 10) | 33 (cách 12) | 11 | 5 |
| 100 | chẵn | 88 | 202 | 88 | 6 |
| 125 | lẻ | 121 | 131 | 121 | 2 |
| 1000 | chẵn | 898 | 2002 | 898 | 51 |

> Bẫy ở `1000`: palindrome chẵn 3 chữ số lớn nhất là `898` (giữa là 9), không phải `888`.

**Bài 2:** `8642`, `642`, `''` (chuỗi rỗng — vì `"5"` chỉ có 1 ký tự); `3`, `4`.

In [21]:
for x, expected in [(10, 1), (21, 5), (100, 6), (125, 2), (1000, 51)]:
    assert nearest_cost_bisect(x) == expected, (x, nearest_cost_bisect(x))
print("Bảng đáp án bài 1 đã được kiểm chứng ✔")

Bảng đáp án bài 1 đã được kiểm chứng ✔


In [22]:
def answer_3_palindromes_of_length(L):
    half_len = (L + 1) // 2
    result = []
    for half in range(10 ** (half_len - 1), 10 ** half_len):
        h = str(half)
        result.append(int(h + h[::-1]) if L % 2 == 0 else int(h + h[-2::-1]))
    return result

assert answer_3_palindromes_of_length(1) == [1, 2, 3, 4, 5, 6, 7, 8, 9]
assert answer_3_palindromes_of_length(2) == [11, 22, 33, 44, 55, 66, 77, 88, 99]
assert len(answer_3_palindromes_of_length(5)) == 900
assert all(is_palindrome(p) for p in answer_3_palindromes_of_length(6))
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [23]:
def answer_4_nearest_cost(x, even_pal, odd_pal):
    arr = even_pal if x % 2 == 0 else odd_pal
    i = bisect_left(arr, x)
    best = float("inf")
    if i < len(arr):
        best = arr[i] - x
    if i > 0:
        best = min(best, x - arr[i - 1])
    return best // 2

for x in range(1, 2000):
    assert answer_4_nearest_cost(x, EVEN_PAL, ODD_PAL) == nearest_cost_bruteforce(x)
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [24]:
PAL_BY_MOD3 = [[p for p in ALL_PAL if p % 3 == r] for r in range(3)]

def answer_5_plus_minus_three(nums):
    total = 0
    for x in nums:
        arr = PAL_BY_MOD3[x % 3]
        i = bisect_left(arr, x)
        best = float("inf")
        if i < len(arr):
            best = arr[i] - x
        if i > 0:
            best = min(best, x - arr[i - 1])
        total += best // 3
    return total

def cost_plus_minus_three_bruteforce(x):
    d = 0
    while True:
        if x - d > 0 and is_palindrome(x - d):
            return d // 3
        if is_palindrome(x + d):
            return d // 3
        d += 3

for x in range(1, 3000):
    assert answer_5_plus_minus_three([x]) == cost_plus_minus_three_bruteforce(x)
print("Đáp án bài 5 PASS")

Đáp án bài 5 PASS


In [25]:
def sieve_primes(N):
    is_prime = [True] * (N + 1)
    is_prime[0] = is_prime[1] = False
    p = 2
    while p * p <= N:
        if is_prime[p]:
            for multiple in range(p * p, N + 1, p):
                is_prime[multiple] = False
        p += 1
    return [i for i in range(2, N + 1) if is_prime[i]]

PRIMES = sieve_primes(10**6 + 100)

def answer_6_nearest_prime_cost(nums):
    total = 0
    for x in nums:
        i = bisect_left(PRIMES, x)
        best = PRIMES[i] - x
        if i > 0:
            best = min(best, x - PRIMES[i - 1])
        total += best
    return total

print(answer_6_nearest_prime_cost([1, 4, 10, 24, 97]))   # 1->2 (1), 4->3/5 (1), 10->11 (1), 24->23 (1), 97 (0)
assert answer_6_nearest_prime_cost([1, 4, 10, 24, 97]) == 4
print("Đáp án bài 6 PASS")

4
Đáp án bài 6 PASS


# 20. Tổng kết

## Kiến thức Python

```python
str(x), s[::-1], s[-2::-1]       # đảo chuỗi, lật bỏ ký tự giữa
from bisect import bisect_left
2_000_000_002                    # gạch dưới phân nhóm chữ số
min(list_of_tuples)              # lấy min kèm thông tin
biến mức module = tính trước một lần
```

## Kỹ thuật thuật toán

1. `±2` giữ **chẵn lẻ** → chỉ xét palindrome cùng chẵn lẻ, chi phí `|x - p| / 2`.
2. Các phần tử **độc lập** → cộng chi phí từng phần tử.
3. Palindrome **hiếm** → **sinh trước** bằng lật nửa đầu.
4. **Bisect** tìm hai hàng xóm, lấy gần hơn.
5. (Khác) Sắp xếp + **hai con trỏ**.

## Công thức nhớ nhanh

> `cost(x) = min(arr[i] - x, x - arr[i-1]) // 2`, với `arr` là danh sách palindrome cùng chẵn lẻ, `i = bisect_left(arr, x)`.

## Độ phức tạp

```text
Precompute: O(P) ≈ 1.2·10^5 palindrome, một lần
Mỗi test:   O(n log P)
Bộ nhớ:     O(P)
```

## Checklist trước khi nộp

- [ ] Tách palindrome **chẵn / lẻ**?
- [ ] Sinh đủ tới `2 000 000 002` (có hàng xóm hai phía)?
- [ ] Kiểm tra biên `i == 0`, `i == len(arr)`?
- [ ] Sinh palindrome **một lần** ở mức module?

# 21. Phiên bản cuối cùng nên nhớ

```python
from bisect import bisect_left

def _build(limit=2_000_000_002):
    even, odd = [], []
    length = 1
    while 10 ** (length - 1) <= limit:
        half_len = (length + 1) // 2
        for half in range(10 ** (half_len - 1), 10 ** half_len):
            h = str(half)
            p = int(h + h[::-1]) if length % 2 == 0 else int(h + h[-2::-1])
            if p > limit:
                break
            (even if p % 2 == 0 else odd).append(p)
        length += 1
    return even, odd

_EVEN, _ODD = _build()

class Solution:
    def minOperations(self, nums: list[int]) -> int:
        total = 0
        for x in nums:
            arr = _EVEN if x % 2 == 0 else _ODD
            i = bisect_left(arr, x)
            best = arr[i] - x if i < len(arr) else float("inf")
            if i > 0:
                best = min(best, x - arr[i - 1])
            total += best // 2
        return total
```

Bài học lớn nhất: khi "đích" là một tập số đặc biệt **thưa thớt**, hãy **liệt kê hết chúng trước**, rồi mỗi truy vấn chỉ còn là một lần binary search.